In [1]:
import numpy as np
import pandas as pd
import glob
import pickle
import cv2
import os
import re

In [2]:
# Function to extract DJI number from a file path
def extract_dji_number(file_path):
    match = re.search(r'DJI_(\d+)', file_path)
    return match.group(1) if match else None

In [3]:
working_directory = '/Volumes/SSD4/processed/Field_Recording_2023'

DATE = ['20230310']#, '20230309', '20230311', '20230312']
SESSION = ['SM_Lek1']#, 'SE_Lek1']
DRONE = ['P1D1', 'P1D2', 'P2D3', 'P2D4', 'P3D5', 'P3D6']

registration_directory = 'SpatialRegistration'
tracking_directory = 'Tracking_Fusion'

In [ ]:
missing_tracks = []  # To store missing tracks.csv paths

for date in DATE:
    for session in SESSION:
        for drone in DRONE:
            registration_path = f"{working_directory}/{registration_directory}/{date}/{session}/{drone}"
            tracking_path = f"{working_directory}/{tracking_directory}/{date}/{session}/{drone}/*{session}_{drone}*"

            # Get frames with corresponding anchors
            anchors = sorted(glob.glob(f"{registration_path}/{date}_{session}_{drone}*_Anchored.csv"))

            # Get tracking folders sorted by DJI video name
            tracking_folders = glob.glob(tracking_path)
            tracking_folders = sorted(tracking_folders, key=lambda x: int(re.search(r'DJI_(\d+)', x).group(1)))

            # Get homography matrices from frame to anchor frame
            homography_matrices = sorted(glob.glob(f"{registration_path}/{date}_{session}_{drone}*_homographies.pkl"))

            # Create dictionaries mapping DJI numbers to file paths
            anchors_dict = {extract_dji_number(file): file for file in anchors}
            tracking_dict = {extract_dji_number(file): file for file in tracking_folders}
            homography_dict = {extract_dji_number(file): file for file in homography_matrices}
            
            # Get sorted list of common DJI numbers
            common_dji_numbers = sorted(set(anchors_dict.keys()) & set(tracking_dict.keys()) & set(homography_dict.keys()), key=int)

            for dji in common_dji_numbers:
                print(date, session, drone, dji)

                try:
                    anchor_file = pd.read_csv(anchors_dict[dji])
                    tracks_csv_path = os.path.join(tracking_dict[dji], 'tracks.csv')
                    
                    if not os.path.isfile(tracks_csv_path):
                        raise FileNotFoundError(f"Missing file: {tracks_csv_path}")
                    
                    tracking_file = pd.read_csv(tracks_csv_path)
                    
                    # Get row count to assert dataframe size at the end
                    original_row_count = len(tracking_file)
                    
                    with open(homography_dict[dji], "rb") as f:
                        homography_file = pickle.load(f)

                    tracking_file = tracking_file.merge(anchor_file[['frame', 'best_anchor_frame']], on='frame', how='left')
                    tracking_file['x'] = tracking_file['bb_left'] + tracking_file['bb_width']/2
                    tracking_file['y'] = tracking_file['bb_top'] + tracking_file['bb_height']/2
                    tracking_file['idx'] = tracking_file['master_track_id']
                    tracking_file = tracking_file.drop_duplicates()

                    tracking_file = tracking_file.loc[:,['frame', 'x', 'y', 'idx', 'class_id', 'class_name', 'best_anchor_frame']]

                    for frame in tracking_file['frame'].unique():
                        if frame in homography_file:
                            H = homography_file[frame]
                            matched_points = tracking_file[tracking_file['frame'] == frame][['idx', 'x', 'y']]

                            if not matched_points.empty:
                                src_pts = np.array(matched_points[['x', 'y']], dtype=np.float32).reshape(-1, 1, 2)
                                transformed_pts = cv2.perspectiveTransform(src_pts, H)

                                tracking_file.loc[tracking_file['frame'] == frame, 'transformed_x'] = transformed_pts[:, 0, 0]
                                tracking_file.loc[tracking_file['frame'] == frame, 'transformed_y'] = transformed_pts[:, 0, 1]
                    
                    output_filename = os.path.basename(anchors_dict[dji]).replace('_Anchored.csv', '_Anchored_trajectories.csv')
                    output_path = os.path.join(os.path.dirname(anchors_dict[dji]), output_filename)
                    tracking_file.to_csv(output_path, index=False, mode='w')

                    # --- ASSERT: compare written file row count to the original tracks.csv row count ---
                    written_df = pd.read_csv(output_path)
                    written_row_count = len(written_df)

                    assert written_row_count == original_row_count, (
                        f"Row count mismatch for {output_path}: written={written_row_count}, "
                        f"original_tracks.csv={original_row_count}"
                    )
                    
                except FileNotFoundError as e:
                    print(f"Warning: {e}")
                    missing_tracks.append(tracking_dict[dji])
                except Exception as e:
                    print(f"Error processing {date}, {session}, {drone}, {dji}: {e}")
                    continue

20230310 SM_Lek1 P1D1 0145
20230310 SM_Lek1 P1D1 0146
20230310 SM_Lek1 P1D1 0147
20230310 SM_Lek1 P1D1 0148
20230310 SM_Lek1 P1D1 0149
20230310 SM_Lek1 P1D1 0150
20230310 SM_Lek1 P1D1 0151
20230310 SM_Lek1 P1D1 0152
20230310 SM_Lek1 P1D1 0153
20230310 SM_Lek1 P1D1 0154
20230310 SM_Lek1 P1D1 0155
20230310 SM_Lek1 P1D1 0156
20230310 SM_Lek1 P1D1 0157
20230310 SM_Lek1 P1D1 0158
20230310 SM_Lek1 P1D1 0159
20230310 SM_Lek1 P1D1 0160
20230310 SM_Lek1 P1D1 0161
20230310 SM_Lek1 P1D1 0162
20230310 SM_Lek1 P1D1 0163
20230310 SM_Lek1 P1D1 0164
20230310 SM_Lek1 P1D1 0165
20230310 SM_Lek1 P1D1 0166
20230310 SM_Lek1 P1D1 0167
20230310 SM_Lek1 P1D1 0168
20230310 SM_Lek1 P1D1 0169
20230310 SM_Lek1 P1D1 0170
20230310 SM_Lek1 P1D1 0171
20230310 SM_Lek1 P1D2 0896
20230310 SM_Lek1 P1D2 0897
20230310 SM_Lek1 P1D2 0898
20230310 SM_Lek1 P1D2 0899
20230310 SM_Lek1 P1D2 0900
20230310 SM_Lek1 P1D2 0901
20230310 SM_Lek1 P1D2 0902
20230310 SM_Lek1 P1D2 0903
20230310 SM_Lek1 P1D2 0904
20230310 SM_Lek1 P1D2 0905
2